# Week08 Capstone Project - Structured Micro Edits

## Strategy
For Week 8, the query policy uses **structured micro-edits**.

By this stage, several functions show a stable promising region. Instead of broad search, this notebook generates a tiny, human-clean neighbourhood around:
- the current best point,
- the latest point,
- the second-best point,
- a weighted merge of those anchors.

The candidate pool is restricted to **micro-moves** on a `0.001` or `0.01` grid, which mirrors the historical submission style: very small edits, stable formatting and reproducible outputs.

A light GP score is used only as a ranking signal among those micro-edits, so the notebook behaves like a disciplined refinement step rather than a fresh global optimiser.


In [1]:
import itertools
import numpy as np
import pandas as pd
import warnings
from pathlib import Path
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel


/Users/sundarid/opt/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [7]:
data_dir = Path('data')
history = pd.read_csv(data_dir / 'weekly_results/Week08.csv')
history.tail(8)


,week,function,d,x1,x2,x3,x4,x5,x6,x7,x8,y
48,7,1,2,0.640,0.640,NaN,NaN,NaN,NaN,NaN,NaN,1.056472
49,7,2,2,0.450,0.750,NaN,NaN,NaN,NaN,NaN,NaN,0.057333
50,7,3,3,0.370,0.570,0.470,NaN,NaN,NaN,NaN,NaN,-0.000248
51,7,4,4,0.398,0.395,0.361,0.442,NaN,NaN,NaN,NaN,0.177731
52,7,5,4,0.940,0.060,0.998,0.996,NaN,NaN,NaN,NaN,3636.434629
53,7,6,5,0.540,0.190,0.640,0.890,0.040,NaN,NaN,NaN,-0.476222
54,7,7,6,0.015,0.182,0.288,0.173,0.397,0.665,NaN,NaN,2.105662
55,7,8,8,0.034,0.437,0.011,0.323,0.989,0.045,0.097,0.702,9.589363


In [8]:
def load_initial(function_id):
    X0 = np.load(data_dir / f'initial_data/F{function_id}_initial_inputs.npy')
    y0 = np.load(data_dir / f'initial_data/F{function_id}_initial_outputs.npy')
    return X0, y0

def load_combined(function_id, history_df):
    X0, y0 = load_initial(function_id)
    rows = history_df[history_df['function'] == function_id].sort_values('week')
    d = int(rows['d'].iloc[0])
    Xw = rows[[f'x{i}' for i in range(1, d + 1)]].to_numpy(dtype=float)
    yw = rows['y'].to_numpy(dtype=float)
    X = np.vstack([X0, Xw])
    y = np.concatenate([y0, yw])
    return X, y, d, rows

def round_grid(x, grid):
    return np.clip(np.round(x / grid) * grid, 0.0, 1.0)

def format_point(x):
    return '-'.join(f'{float(v):.6f}' for v in x)


In [9]:
def build_gp(seed, d):
    kernel = (
        ConstantKernel(1.0, (0.1, 10.0))
        * RBF(length_scale=np.ones(d), length_scale_bounds=(0.01, 1.5))
        + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-8, 1e-2))
    )
    return GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        random_state=seed,
        n_restarts_optimizer=1,
    )

micro_cfg = {
    1: {'step': 0.010, 'grid': 0.010},
    2: {'step': 0.030, 'grid': 0.010},
    3: {'step': 0.010, 'grid': 0.010},
    4: {'step': 0.001, 'grid': 0.001},
    5: {'step': 0.001, 'grid': 0.001},
    6: {'step': 0.010, 'grid': 0.010},
    7: {'step': 0.001, 'grid': 0.001},
    8: {'step': 0.001, 'grid': 0.001},
}


In [10]:
def build_micro_candidates(best_x, latest_x, second_x, d, step, grid):
    candidates = []
    anchors = [
        best_x,
        latest_x,
        second_x,
        (2 * best_x + latest_x) / 3.0,
        (best_x + latest_x) / 2.0,
        (best_x + second_x) / 2.0,
    ]
    for a in anchors:
        candidates.append(round_grid(a, grid))

    for i in range(d):
        for sign in (-1.0, 1.0):
            x = best_x.copy()
            x[i] += sign * step
            candidates.append(round_grid(x, grid))

    for i, j in itertools.combinations(range(min(d, 4)), 2):
        for si in (-1.0, 1.0):
            for sj in (-1.0, 1.0):
                x = best_x.copy()
                x[i] += si * step
                x[j] += sj * step
                candidates.append(round_grid(x, grid))

    return np.unique(np.array(candidates), axis=0)


In [11]:
def suggest_week8_point(function_id, history_df, seed=700):
    X, y, d, rows = load_combined(function_id, history_df)
    best_idx = int(np.argmax(y))
    second_idx = int(np.argsort(y)[-2])
    best_x = X[best_idx].copy()
    second_x = X[second_idx].copy()
    latest_x = rows[[f'x{i}' for i in range(1, d + 1)]].to_numpy(dtype=float)[-1].copy()

    cfg = micro_cfg[function_id]
    step, grid = cfg['step'], cfg['grid']
    candidates = build_micro_candidates(best_x, latest_x, second_x, d, step, grid)

    # keep micro-edits novel enough
    dist = np.sqrt(((candidates[:, None, :] - X[None, :, :]) ** 2).sum(axis=2)) / np.sqrt(d)
    min_dist = dist.min(axis=1)
    keep = min_dist >= (0.0005 if grid == 0.001 else 0.005)
    candidates = candidates[keep]
    min_dist = min_dist[keep]

    gp = build_gp(seed + function_id, d)
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', category=ConvergenceWarning)
        gp.fit(X, y)

    mean, std = gp.predict(candidates, return_std=True)
    best_dist = np.sqrt(((candidates - best_x) ** 2).sum(axis=1)) / np.sqrt(d)
    latest_dist = np.sqrt(((candidates - latest_x) ** 2).sum(axis=1)) / np.sqrt(d)
    score = mean + 0.15 * std - 0.20 * best_dist - 0.10 * latest_dist + 0.02 * min_dist

    idx = int(np.argmax(score))
    x_best = candidates[idx]
    return {
        'function': function_id,
        'd': d,
        'x': x_best,
        'formatted': format_point(x_best),
        'step': step,
        'grid': grid,
        'mean_pred': float(mean[idx]),
        'std_pred': float(std[idx]),
        'min_dist': float(min_dist[idx]),
        'best_anchor': format_point(best_x),
        'latest_anchor': format_point(latest_x),
    }


In [ ]:
results = []
for function_id in range(1, 9):
    result = suggest_week8_point(function_id, history)
    results.append(result)
    print(f'Function {function_id}: {result["formatted"]}')
    print(f'  step={result["step"]:.3f}, grid={result["grid"]:.3f}, mean={result["mean_pred"]:.6f}, std={result["std_pred"]:.6f}')
    print(f'  best_anchor={result["best_anchor"]}')
    print(f'  latest_anchor={result["latest_anchor"]}')
    print()
